---

# FINAL EXAM MACHINE LEARNING I

date: "30 April 2025"

author: 
Fernando San Segundo Badahona & Néstor Rodríguez Pérez


---


`fill your name here`


::: {.callout-important icon=false}

ChatGPT and other AI assistants are not allowed, but you can access Internet and your course files; do not hesitate using them, as it will probably be the best way to write the necessary code, check the documentation or solve coding issues.

:::

::: {.callout-warning icon=false}

**Please refrain from spending time in unrequested code or graphs.**

They will not count towards your grade and will only leave you with less time for the relevant parts of the exam.

:::

# **Milk production dataset** (4 points)

The data is provided in the file 

`milk_production.csv`

The meaning of the variables is as follows:

+ The `DATE` of the observation.
+ `MILK` production in thousands of tons. 
 

Your goal is to compare different forecasting algorithms to predict the output variable `MILK`. 



::: {.callout-tip icon="false}

### Question 0:  Exploratory Data Analysis (EDA) and data preparation (1 point). 

- Load the dataset using pandas. 
- Perform data exploration and thorough cleaning of the dataset.
- Visualize the time series. Explore the trend and seasonality or seasonalities of this time series.
- Split the data into training and testing sets. **The test set is the last 12 rows of the dataset**. For validation in the training set, use an initial window consisting of **75% of the training set**, and choose the step size to get **10 folds**.
- Preprocess the data as needed. 
- The validation method for all the models below should use a **rolling forecast origin**, with a forecasting horizon of 1 timestep and with the refit strategy. We recommend using the tools provided by sktime for this and the following questions.

:::

In [31]:
import subprocess

def pip_install(package):
    result = subprocess.run(
        ["pip", "install", package],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        print(f"Error installing {package}: {result.stderr}")
    else:
        print(f"Successfully installed {package}")

pip_install("fpppy")
pip_install("tsfeatures")
pip_install("pyreadr")

In [32]:
# 1. System and Environment Setup
import os
import sys
import warnings
import random
import datetime
from tqdm import tqdm

# Set environment variables and ignore specific warnings
os.environ["NIXTLA_ID_AS_COL"] = "true"
warnings.filterwarnings(
    "ignore", 
    category=UserWarning, 
    message=".*FigureCanvasAgg is non-interactive.*"
)

# 2. Core Data Science Stack
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose

# 3. Time Series & Machine Learning
import pmdarima as pmd
from sktime.datasets import load_airline
from sktime.utils.plotting import plot_series as plot_series_sktime

# 4. Custom Utilities
# from fc_4_1_utils.utils import *
from fpppy.utils import plot_series as plot_series_fpp
from utilsforecast.plotting import plot_series as plot_series_utils

# 5. Global Reproducibility & Display Settings
np.random.seed(1)
random.seed(1)
np.set_printoptions(suppress=True)

pd.set_option("max_colwidth", 100)
pd.set_option("display.precision", 3)

# 6. Plotting Configuration
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from cycler import cycler

# Jupyter Magic
%matplotlib inline
%config InlineBackend.figure_format = 'png' # High-res plots

# Style and Aesthetics
plt.style.use("ggplot")
sns.set_style("whitegrid")

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "figure.constrained_layout.use": True,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "legend.title_fontsize": 10,
})

# Custom color cycle (Example: monochromatic black as per your original)
mpl.rcParams['axes.prop_cycle'] = cycler(color=["#000000", "#000000"])

In [33]:
%cd "/wd/00_previous_exams/Final2025/2025_FinalExam"

/wd/00_previous_exams/Final2025/2025_FinalExam


In [34]:
!pwd

/wd/00_previous_exams/Final2025/2025_FinalExam


In [35]:
!ls

2025_FinalExam_Unsupervised_statement.ipynb  fc_4_3_utils   milk_production.csv
2025_Final_Forecasting_statement.ipynb	     gdpr_data.csv


In [ ]:
# Note that ; is used to separate the columns
df = pd.read_csv('milk_production.csv',  sep=';')
df.head()

,DATE,MILK
0,1962-01-31,589.0
1,1962-02-28,561.0
2,1962-03-31,640.0
3,1962-04-30,656.0
4,1962-05-31,727.0


In [37]:
df["DATE"] = pd.to_datetime(df["DATE"], format="%Y-%m-%d")
df.head()

,DATE,MILK
0,1962-01-31,589.0
1,1962-02-28,561.0
2,1962-03-31,640.0
3,1962-04-30,656.0
4,1962-05-31,727.0


In [38]:
# Let us change the date to be the first day of the month
df['DATE'] = df['DATE'].dt.to_period('M').dt.to_timestamp()
df.head()   

,DATE,MILK
0,1962-01-01,589.0
1,1962-02-01,561.0
2,1962-03-01,640.0
3,1962-04-01,656.0
4,1962-05-01,727.0


In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   DATE    168 non-null    datetime64[ns]
 1   MILK    167 non-null    float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 2.8 KB



::: {.callout-tip icon="false}

### Question 1 : Identification and Fitting Process of a *seasonal last* naive model as baseline (0.5 points)

+ Fit **a seasonal last naive** model to the data.
+ Evaluate the performance of the model using MAPE. Thie will be the performance measure used for the selection between the models. 
+ Visualize the results of the baseline models.


:::


::: {.callout-tip icon="false}

### Question 2: SARIMA model (1.5 points)

- Select and diagnose a SARIMA model for the time series.
- Evaluate the performance of this model in validation and test.
- Visualize the predictions of this model.

:::


::: {.callout-tip icon="false}

### Question 3 : Gradient boosting model  (0.5 points)

- Fit a gradient boosting model. Use grid search to select the hyperparameters for this model.
- Evaluate the performance of this model in validation and test.
- Visualize the predictions of this model.

:::


::: {.callout-tip icon="false}

### Question 4: Final model comparison and conclusions (0.5 points)

- Summarize your findings from the comparative analysis of these forecasting models. 

:::